In [1]:
# Imports and Configurations
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import time

# Clustering and distance metrics
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score

# Parallelization
import multiprocessing as mp

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

np.random.seed(42)

# Configuration
BENCHMARK      = '^GSPC'
RISK_FREE_RATE = 0.045
START_DATE     = '2005-01-01'
END_DATE       = '2026-01-01'
ROLLING_WINDOW = 52
MIN_HIST       = 400
K_VALUES       = [2, 4, 7]
N_CORES        = 14

# Three-period out-of-sample validation framework (20-year-window)
TRAIN_START = '2005-01-01'
TRAIN_END   = '2010-12-31'   # 6 years
VAL_START   = '2011-01-01'
VAL_END     = '2017-12-31'   # 7 years
TEST_START  = '2018-01-01'
TEST_END    = '2025-12-31'   # 7 years

# Strategy Parameters (tightened version of Notebook 6)
ZSCORE_WINDOW   = 52
VOL_WINDOW      = 26
VOL_THRESHOLD   = 1.5
HIGH_VOL_ENTRY  = 2.25  # was 2.5
LOW_VOL_ENTRY   = 1.25  # was 1.5
STANDARD_ENTRY  = 1.75  # was 2.0
EXIT_THRESHOLD  = 0.5
STOP_LOSS       = 3.25  # was 3.5

print('Notebook 7: DTW vs OCP vs TOP Clustering - S&P 500, 20-Year Three-Period Framework')
print(f'Train:  {TRAIN_START} to {TRAIN_END} (6 years)')
print(f'Val:    {VAL_START} to {VAL_END} (7 years)')
print(f'Test:   {TEST_START} to {TEST_END} (7 years)')

Notebook 7: DTW vs OCP vs TOP Clustering - S&P 500, 20-Year Three-Period Framework
Train:  2005-01-01 to 2010-12-31 (6 years)
Val:    2011-01-01 to 2017-12-31 (7 years)
Test:   2018-01-01 to 2025-12-31 (7 years)


In [4]:
import re

# Retrieving S&P 500 constitutents as of January 1, 2005

# Survivorship-bias free source
hist_url = (
    'https://raw.githubusercontent.com/fja05680/sp500/master/'
    'S%26P%20500%20Historical%20Components%20%26%20Changes.csv'
)

hist = pd.read_csv(hist_url)
hist['date'] = pd.to_datetime(hist['date'])

# Targeting the row closest to our date
target_date = pd.Timestamp('2005-01-01')
snapshot_row = hist[hist['date'] <= target_date].sort_values('date').iloc[-1]

print(f'Using snapshot date: {snapshot_row["date"].date()}')

sp500_tickers_2005 = sorted(snapshot_row['tickers'].split(','))

raw_tickers = sorted(snapshot_row['tickers'].split(','))

# Stipping the dataset's own "-YYYYMM" delisting_date suffix convention
suffix_pattern = re.compile(r'-\d{6}$')

sp500_tickers_2005 = sorted(set(
    suffix_pattern.sub('', t.strip()).replace('.','-')
    for t in raw_tickers
))

n_suffixed = sum(1 for t in raw_tickers if suffix_pattern.search(t.strip()))
print(f'Tickers with a delisting-date suffix in the source data: {n_suffixed}')
print(f'S&P 500 constituents as of {target_date.date()}: {len(sp500_tickers_2005)} stocks')
print(f'\nFirst 10 tickers: {sp500_tickers_2005[:10]}')

Using snapshot date: 2004-12-30
Tickers with a delisting-date suffix in the source data: 179
S&P 500 constituents as of 2005-01-01: 495 stocks

First 10 tickers: ['A', 'AABA', 'AAPL', 'ABC', 'ABI', 'ABKFQ', 'ABS', 'ABT', 'ACS', 'ACV']


In [ ]:
# Downloading price data for this S&P 500 205 universe
import yfinance as yf

print(f'Downloading price data for {len(sp500_tickers_2005)} stocks...')
print(f'Period: {START_DATE} to {END_DATE}, weekly (Wednesday), adjusted close\n')

raw = yf.download(
    sp500_tickers_2005,
    start=START_DATE,
    end=END_DATE,
    interval='1wk',
    auto_adjust=True,
    progress=True
)

price_data = raw['Close']
price_data = price_data.resample('W-WED').last()
price_data = price_data.dropna(how='all')

# Dropping any tickers without complete history across the full history
before_drop = price_data.shape[1]
price